# Fine-Tune Ambulance Detection (Overhead Camera Angle)

**Goal:** Continue fine-tuning the existing `vehicle_detection_yolov8l_ambulance.pt` model
so it also detects ambulances from **overhead / top-down camera angles**.

The model currently detects ambulances well from horizontal angles.
This notebook adds overhead-angle training data (COCO format, 59 images)
without losing existing detection capability.

| Model file | Purpose |
|---|---|
| `vehicle_detection_yolov8l_ambulance.pt` | Ambulance detection (dedicated fine-tuned model) |

**Strategy:** Freeze backbone layers to preserve existing features.
Low learning rate + short training to add overhead angle knowledge.

---
**Estimated GPU time:** ~10-20 minutes (Kaggle P100 / T4)  
**Output:** `vehicle_detection_yolov8l_ambulance.pt` (drop-in replacement)


## REQUIRED: Datasets to Attach in Kaggle

Before running this notebook, attach **two datasets** via `+ Add data`:

---
### Dataset 1 - Overhead Ambulance Images (COCO format)
**Slug:** `ambulance-overhead-coco`

Upload the folder `dataset/ambulance.v1i.coco/` as a Kaggle dataset.
It should contain:
```
ambulance-overhead-coco/
  train/
    _annotations.coco.json   (59 images, 60 annotations)
    amb1_jpg.rf.xxx.jpg
    ...                      (59 .jpg files, 640x640)
```

---
### Dataset 2 - Pre-trained Ambulance Model Weights
**Slug:** `ambulance-model-weights`

Upload the single file:
```
video_detection/vehicle_detection_yolov8l_ambulance.pt
```

If not attached, the notebook falls back to public `yolov8l.pt` (COCO weights),
which still works but starts without existing ambulance knowledge.


---
## Step 1 - Setup & Installation


In [ ]:
!pip install ultralytics>=8.0.0 -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow on CPU.")


In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from collections import Counter

import yaml
from ultralytics import YOLO

# ============================================================
# INPUT PATHS - update if your Kaggle dataset slugs differ
# ============================================================
COCO_DATASET_DIR   = Path('/kaggle/input/ambulance-overhead-coco/train')
PRETRAINED_WEIGHTS  = Path('/kaggle/input/ambulance-model-weights/vehicle_detection_yolov8l_ambulance.pt')
# ============================================================

WORK_DIR = Path('/kaggle/working')

# Confirm inputs
print("Input checks:")
for label, path in [
    ("COCO dataset dir",   COCO_DATASET_DIR),
    ("Annotations JSON",   COCO_DATASET_DIR / '_annotations.coco.json'),
    ("Pre-trained weights", PRETRAINED_WEIGHTS),
]:
    status = 'OK' if path.exists() else 'NOT FOUND'
    print(f"  [{status}] {label}: {path}")

# Fall back to public COCO weights if custom weights not attached
if not PRETRAINED_WEIGHTS.exists():
    PRETRAINED_WEIGHTS = 'yolov8l.pt'
    print(f"\nFalling back to: {PRETRAINED_WEIGHTS} (downloads automatically)")
else:
    PRETRAINED_WEIGHTS = str(PRETRAINED_WEIGHTS)


---
## Step 2 - Convert COCO to YOLO Format + Train/Val Split

The dataset is in COCO JSON format. YOLO needs per-image `.txt` label files
with format: `class_id center_x center_y width height` (normalized 0-1).

COCO annotation `category_id=1` (ambulance) maps to YOLO **class 4** in our 5-class model.

We split the 59 images into **train (85%)** and **val (15%)**.


In [ ]:
DATASET_DIR = WORK_DIR / 'ambulance_overhead'

# Load COCO annotations
coco_json = COCO_DATASET_DIR / '_annotations.coco.json'
with open(coco_json) as f:
    coco = json.load(f)

print(f"COCO dataset: {len(coco['images'])} images, {len(coco['annotations'])} annotations")
print(f"Categories: {coco['categories']}")

# Build lookup: image_id -> image info
img_lookup = {img['id']: img for img in coco['images']}

# Build lookup: image_id -> list of annotations
ann_lookup = {}
for ann in coco['annotations']:
    ann_lookup.setdefault(ann['image_id'], []).append(ann)

# Convert COCO bbox [x, y, w, h] (absolute) to YOLO [cx, cy, w, h] (normalized)
def coco_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh

# Shuffle and split
all_images = list(coco['images'])
random.seed(42)
random.shuffle(all_images)

n_val = max(1, int(len(all_images) * 0.15))  # ~9 images for val
val_images = all_images[:n_val]
train_images = all_images[n_val:]

print(f"\nSplit: {len(train_images)} train / {len(val_images)} val")

# Convert and write
for split_name, split_imgs in [('train', train_images), ('val', val_images)]:
    img_dir = DATASET_DIR / split_name / 'images'
    lbl_dir = DATASET_DIR / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_info in split_imgs:
        # Copy image
        src_path = COCO_DATASET_DIR / img_info['file_name']
        if not src_path.exists():
            print(f"  WARNING: missing {src_path}")
            continue
        shutil.copy2(src_path, img_dir / img_info['file_name'])

        # Convert annotations to YOLO format
        img_w = img_info['width']
        img_h = img_info['height']
        anns = ann_lookup.get(img_info['id'], [])

        lines = []
        for ann in anns:
            # COCO category_id 1 = ambulance -> YOLO class 4
            if ann['category_id'] != 1:
                continue
            cx, cy, nw, nh = coco_to_yolo(ann['bbox'], img_w, img_h)
            lines.append(f"4 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        lbl_path = lbl_dir / (Path(img_info['file_name']).stem + '.txt')
        lbl_path.write_text('\n'.join(lines))

    print(f"  {split_name}: {len(split_imgs)} images converted")


---
## Step 3 - Create dataset.yaml & Verify Labels


In [ ]:
# Create dataset.yaml for YOLO training
dataset_config = {
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'nc': 5,
    'names': {
        0: 'car',
        1: 'truck',
        2: 'motorcycle',
        3: 'bus',
        4: 'ambulance',
    },
}

DATASET_YAML = WORK_DIR / 'ambulance_overhead.yaml'
with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"dataset.yaml saved to: {DATASET_YAML}")
print()
print(DATASET_YAML.read_text())


In [ ]:
# Verify label distribution after conversion
print("Label distribution per split:")
print("=" * 50)

for split in ['train', 'val']:
    class_counter = Counter()
    lbl_dir = DATASET_DIR / split / 'labels'
    empty_count = 0
    for lbl_file in sorted(lbl_dir.glob('*.txt')):
        content = lbl_file.read_text().strip()
        if not content:
            empty_count += 1
            continue
        for line in content.splitlines():
            parts = line.strip().split()
            if parts:
                class_counter[int(parts[0])] += 1

    total_boxes = sum(class_counter.values())
    n_images = len(list((DATASET_DIR / split / 'images').glob('*')))
    print(f"\n{split.upper()} ({n_images} images, {total_boxes} boxes):")
    for cid in sorted(class_counter.keys()):
        name = {0: 'car', 1: 'truck', 2: 'motorcycle', 3: 'bus', 4: 'ambulance'}.get(cid, f'class_{cid}')
        count = class_counter[cid]
        print(f"  [{cid}] {name:12s}: {count:5d}")
    if empty_count:
        print(f"  (empty labels: {empty_count})")

# Sanity check
all_classes = set()
for split in ['train', 'val']:
    for lbl_file in (DATASET_DIR / split / 'labels').glob('*.txt'):
        for line in lbl_file.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                all_classes.add(int(parts[0]))

print(f"\nClass IDs found: {sorted(all_classes)}")
if all_classes == {4}:
    print("OK: Only class 4 (ambulance) present.")


In [ ]:
# Visual preview: draw YOLO labels on a few images
import cv2
import matplotlib.pyplot as plt

preview_imgs = sorted((DATASET_DIR / 'train' / 'images').glob('*'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, preview_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    lbl_path = DATASET_DIR / 'train' / 'labels' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                _, cx, cy, bw, bh = map(float, parts)
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, 'ambulance', (x1, max(y1 - 5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    ax.imshow(img)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis('off')

for ax in axes[len(preview_imgs):]:
    ax.axis('off')

plt.suptitle('COCO -> YOLO Label Conversion Preview', fontsize=13)
plt.tight_layout()
plt.show()


---
## Step 4 - Fine-tune Ambulance Model

**Strategy for continued fine-tuning with small dataset (59 images):**

- `freeze=10` - freeze backbone to preserve existing ambulance + vehicle features
- `epochs=60` with `patience=15` - enough to learn overhead angles, early stop prevents overfit
- `lr0=0.002` - very low LR since model already knows ambulances, just adding new angles
- Strong augmentation (mosaic, scale, copy_paste) to maximize the small dataset
- `batch=8` - small batches work better with small datasets

**This only modifies `vehicle_detection_yolov8l_ambulance.pt`.**
The general vehicle model and accident/traffic classifiers are untouched.


In [ ]:
print(f"Loading base model: {PRETRAINED_WEIGHTS}")
model = YOLO(PRETRAINED_WEIGHTS)

# Print class names from the loaded model
print(f"Model classes: {model.names}")


In [ ]:
print("Starting fine-tuning (overhead angle ambulance data)...")
print("=" * 60)

results = model.train(
    data=str(DATASET_YAML),

    # -- Training duration --
    epochs=60,
    patience=15,

    # -- Batch & resolution --
    batch=8,
    imgsz=640,

    # -- Fine-tuning settings --
    freeze=10,          # Freeze backbone - preserve existing features
    lr0=0.002,          # Very low LR - model already knows ambulances
    lrf=0.01,
    warmup_epochs=3,

    # -- Loss weights --
    cls=1.5,
    box=7.5,
    dfl=1.5,

    # -- Optimizer --
    optimizer='AdamW',
    weight_decay=0.0005,

    # -- Augmentation (aggressive for small dataset) --
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,        # No rotation - cameras are fixed
    translate=0.15,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    copy_paste=0.2,     # Higher copy_paste for small dataset
    mixup=0.1,

    # -- Output --
    device=0,
    workers=2,
    project='ambulance_overhead_training',
    name='ambulance_ft_overhead',
    exist_ok=True,
    pretrained=True,
    verbose=True,
    seed=42,
)

print("\nFine-tuning complete!")


---
## Step 5 - Evaluate Model


In [ ]:
from IPython.display import Image, display

BEST_PT = Path('ambulance_overhead_training/ambulance_ft_overhead/weights/best.pt')
print(f"Best model: {BEST_PT} -- exists: {BEST_PT.exists()}")

best_model = YOLO(str(BEST_PT))

val_results = best_model.val(
    data=str(DATASET_YAML),
    split='val',
    batch=8,
    imgsz=640,
    verbose=False,
)

print("\n" + "=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
print(f"mAP50    : {val_results.box.map50:.4f}")
print(f"mAP50-95 : {val_results.box.map:.4f}")

print("\nPer-class AP50:")
class_names = ['car', 'truck', 'motorcycle', 'bus', 'ambulance']
for i, name in enumerate(class_names):
    if hasattr(val_results.box, 'ap50') and i < len(val_results.box.ap50):
        ap = val_results.box.ap50[i]
        bar = '#' * int(ap * 30)
        print(f"  [{i}] {name:12s}: {ap:.4f}  {bar}")


In [ ]:
# Spot-check: inference on validation images
import cv2
import matplotlib.pyplot as plt

val_dir = DATASET_DIR / 'val' / 'images'
val_frames = sorted(val_dir.glob('*'))[:6]

if val_frames:
    n = len(val_frames)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for ax, img_path in zip(axes, val_frames):
        res = best_model(str(img_path), conf=0.25, verbose=False)
        out = cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(out)
        ax.set_title(img_path.name[:35], fontsize=8)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle('Validation Samples -- Overhead Ambulance Detections', fontsize=13)
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'ambulance_overhead_predictions.png', dpi=150)
    plt.show()
else:
    print("No validation frames found.")


In [ ]:
# Show training curves
results_dir = Path('ambulance_overhead_training/ambulance_ft_overhead')

for img_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(img_name)
        display(Image(filename=str(img_path), width=800))


---
## Step 6 - Export & Package for Download


In [ ]:
import zipfile

OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

# Copy best weights - this is the drop-in replacement
shutil.copy2(BEST_PT, OUTPUT_DIR / 'vehicle_detection_yolov8l_ambulance.pt')

# Copy last weights as backup
last_pt = BEST_PT.with_name('last.pt')
if last_pt.exists():
    shutil.copy2(last_pt, OUTPUT_DIR / 'vehicle_detection_yolov8l_ambulance_last.pt')

# Copy training artifacts
results_dir = Path('ambulance_overhead_training/ambulance_ft_overhead')
for artifact in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    src = results_dir / artifact
    if src.exists():
        shutil.copy2(src, OUTPUT_DIR / artifact)

spot = WORK_DIR / 'ambulance_overhead_predictions.png'
if spot.exists():
    shutil.copy2(spot, OUTPUT_DIR / 'ambulance_overhead_predictions.png')

# Zip everything
zip_path = WORK_DIR / 'vehicle_detection_ambulance_overhead_update.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.iterdir():
        zf.write(f, f.name)

print(f"Package ready: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1e6:.1f} MB")
print("\nFiles in package:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")


---
## Step 7 - Integration Instructions

After downloading `vehicle_detection_ambulance_overhead_update.zip` from the Kaggle Output tab:

### 1. Replace the ambulance model file
```
Traffic-Platform/
  video_detection/
    vehicle_detection_yolov8l_ambulance.pt   <-- replace with downloaded file
```

**Do NOT replace** `vehicle_detection_yolov8l.pt` (general vehicle model) or any other model.

### 2. Test with the demo script
```bash
cd Traffic-Platform
python demo.py --img imgs/e.jpg imgs/w.jpg imgs/s.jpg imgs/n.jpg -o results/
python demo.py --video videos/test.mp4 -o results/
```

### 3. If ambulance AP50 < 0.5
- Reduce `freeze` from 10 to 5 and retrain
- Lower ambulance threshold in `video_detection/config/config.yaml` to 0.20
- Add more overhead-angle ambulance images to the dataset


In [ ]:
print("=" * 60)
print("FINE-TUNING COMPLETE -- SUMMARY")
print("=" * 60)

print(f"\nBase model     : {PRETRAINED_WEIGHTS}")
print(f"Dataset        : Overhead ambulance COCO v1 (59 images)")
print(f"Epochs         : 60 (with early stopping, patience=15)")
print(f"Image size     : 640")
print(f"Frozen layers  : 10 (backbone preserved)")

print(f"\nmAP50 (all)      : {val_results.box.map50:.4f}")
print(f"mAP50-95 (all)   : {val_results.box.map:.4f}")
if hasattr(val_results.box, 'ap50') and len(val_results.box.ap50) > 4:
    print(f"AP50 (ambulance) : {val_results.box.ap50[4]:.4f}")

print(f"\nOutput package   : {zip_path}")
print("\nDownload from Kaggle Output tab and replace:")
print("  video_detection/vehicle_detection_yolov8l_ambulance.pt")
